In [1]:
import pandas as pd
import os
from pydub import AudioSegment
from pydub.playback import play

def get_time(row):
    """
    this function returns the start and end time for the interval in milliseconds

    Parameters
    ----------
    row : pandas Series
        a row of the dataframe containing the filename, start_datetime and end_datetime
    Returns
    -------
    Start : int
        the start time of the interval in seconds
    End : int
        the end time of the interval in seconds
    """

    initial_time = row["filename"][11:-4]
    initial_time = int(initial_time[:2])*3600 + int(initial_time[3:5])*60 + int(initial_time[6:8])
    initial_time = initial_time * 1000

    Start = row['start_datetime'][11:26]
    End = row['end_datetime'][11:26]
    Start=int(float(Start[:2]))*3600 + int(float(Start[3:5]))*60 + int(float(Start[6:8])) + float("0." + Start[9:])
    End=int(float(End[:2]))*3600 + int(float(End[3:5]))*60 + int(float(End[6:8])) + float("0." + End[9:])

    Start *= 1000
    End *= 1000

    Start = Start - initial_time
    End = End - initial_time

    return Start, End


os.makedirs('waves', exist_ok=False)

i = 0 # this iterator is used for naming uniqueness, since in our first run and after 2 hours, alot were overwritten(exactly 32,142)
for file in os.scandir('biodcase_development_set/train/annotations'):
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file.path)
        
        for index, row in df.iterrows():
            i+=1
            wave_name=row['filename']
            folder_name = row['dataset']
            wav_file = AudioSegment.from_file(file='biodcase_development_set/train/audio/' +folder_name+ '/' +wave_name, format="wav")
           
            Start, End = get_time(row)
            newAudio=wav_file[Start:End]

            newfilename = row["annotation"] + '_' + str(row["low_frequency"]) + '_' + str(row["high_frequency"]) + "_" + str(i) + ".wav"
            newAudio.export('waves/' + newfilename, format="wav")

In [2]:
csv_lines = 0
for file in os.scandir('biodcase_development_set/train/annotations'):
    if file.is_file() and file.name.endswith('.csv'):
        csv_lines += len(pd.read_csv(file.path))

print("number of lines in train csv files is: ", csv_lines)

number of lines in train csv files is:  58570


In [3]:
wave_events = 0
for file in os.scandir('waves'):
    if file.is_file() and file.name.endswith('.wav'):
        wave_events += 1

print("number of wave events cutted out: ", wave_events)

number of wave events cutted out:  58570


# implement spectrogram

In [ ]:
# will run this in test.py file, since in REPL the same python script uses much more CPU and stacks RAM nonestop increasingly
# since by default the notebook wants to print out the result (the calculated spectrogram)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile


WAV_DIR = Path("./waves")
OUTPUT_DIR = Path("./spectrograms")

NFFT = 512
WINSIZE = 256
OVERLAP_PERCENT = 98
SCALE = "default"


def convert_wav_to_float(data: np.ndarray) -> np.ndarray:
    """Match the notebook conversion so plotting behaves consistently across WAV dtypes."""
    if data.dtype == np.uint8:
        data = (data - 128) / 128.0
    elif data.dtype == np.int16:
        data = data / 32768.0
    elif data.dtype == np.int32:
        data = data / 2147483648.0
    return data.astype(np.float32, copy=False)


def parse_filename(stem: str) -> tuple[str, float, float]:
    """Expected pattern: <label>_<low_freq>_<high_freq>.wav"""
    parts = stem.split("_")
    if len(parts) != 3:
        raise ValueError(f"Unexpected filename format: {stem}")

    label, low_str, high_str = parts
    return label, float(low_str), float(high_str)


def load_wav_mono(path: Path) -> tuple[int, np.ndarray]:
    sample_rate, data = wavfile.read(path)
    data = convert_wav_to_float(data)

    # Convert multi-channel WAV to mono for a single spectrogram image.
    if data.ndim > 1:
        data = data.mean(axis=1)

    return sample_rate, data


def save_spectrogram(wav_path: Path) -> None:
    _, low_freq, high_freq = parse_filename(wav_path.stem)
    sample_rate, signal = load_wav_mono(wav_path)

    noverlap = int(round(WINSIZE * (OVERLAP_PERCENT / 100.0)))
    noverlap = min(noverlap, WINSIZE - 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.specgram(
        signal,
        NFFT=WINSIZE,
        Fs=sample_rate,
        noverlap=noverlap,
        pad_to=NFFT,
        scale=SCALE,
        cmap="cividis",
    )

    low = min(low_freq, high_freq)
    high = max(low_freq, high_freq)
    ax.set_ylim(low, high)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(wav_path.stem)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"{wav_path.stem}.png"
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)


def main() -> None:
    wav_files = sorted(WAV_DIR.glob("*.wav"))
    for wav_path in wav_files:
        save_spectrogram(wav_path)


if __name__ == "__main__":
    main()

# Summary:
# - Iterates over every WAV in ./waves.
# - Parses filename as <label>_<low_freq>_<high_freq>.wav.
# - Builds one spectrogram per WAV using nfft=512, winsize=256, overlap=98, scale=default.
# - Uses the parsed low/high frequencies as the y-axis limits.
# - Saves each image to ./spectrograms with the same base filename and .png extension.

In [1]:
%ls spectrograms/ | wc -l

58570


This command prints all the spectrogram image file directories inside ./spectrograms, then i pipe the output to wc -l which will count only the lines since we added the line flag

### Model